In [62]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
products_df = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
sellers_df = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
product_category_name_translation_df = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")


In [63]:
#checking for nulls in each dataset

for name, d in {"products": products_df,
                   "sellers": sellers_df,
                   "translation": product_category_name_translation_df}.items():
    print(name, d.shape)
    print(d.isna().sum()[d.isna().sum() > 0], "\n")

display(products_df.head())
display(product_category_name_translation_df.head())
display(sellers_df.head())


products (32951, 9)
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64 

sellers (3095, 4)
Series([], dtype: int64) 

translation (71, 2)
Series([], dtype: int64) 



,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [64]:
products_df[products_df.isna().any(axis=1)]
nulls = products_df['product_category_name'].isna().sum()
print(f"Number of nulls in product_category_name: {nulls}")
null_indexis = products_df[products_df['product_category_name'].isna()].index
null_rows = products_df.loc[null_indexis]
display(null_rows.isna().sum())

Number of nulls in product_category_name: 610


product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64

### found that there are 610 nulls in each of the name , length and description columns The nulls are all in the same rows
these rows are real orders that may be sold, dropping them could corrupt the other sales dataframe so i kept them for now\
instead of dropping them i would change the name to unknown


In [65]:
products_df["product_category_name"] = products_df["product_category_name"].fillna("unknown")
products_df["product_category_name"].isna().sum()
#changed the null values in the product name to "unknown" instead of dropping rows

products_df['has_listing_category'] = products_df['product_name_lenght'].notna()
#added a flag to differentiate between products with and without listing category

In [66]:
display(products_df[products_df['product_category_name'] == 'unknown'].head())
display(products_df.head())

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,has_listing_category
105,a41e356c76fab66334f36de622ecbd3a,unknown,NaN,NaN,NaN,650.0,17.0,14.0,12.0,False
128,d8dee61c2034d6d075997acef1870e9b,unknown,NaN,NaN,NaN,300.0,16.0,7.0,20.0,False
145,56139431d72cd51f19eb9f7dae4d1617,unknown,NaN,NaN,NaN,200.0,20.0,20.0,20.0,False
154,46b48281eb6d663ced748f324108c733,unknown,NaN,NaN,NaN,18500.0,41.0,30.0,41.0,False
197,5fb61f482620cb672f5e586bb132eae9,unknown,NaN,NaN,NaN,300.0,35.0,7.0,12.0,False


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,has_listing_category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,True
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,True
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,True
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,True
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,True


In [67]:
products_df[products_df['product_weight_g'].isna()]
#here i located the rows with missing dimensions and weight, dropping them would delete the data of a sold product
#so i will keep them 


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,has_listing_category
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN,True
18851,5eb564652db742ff8f28759cd8d2652a,unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


In [68]:
print(f"Number of duplicated products in the dataset is : {products_df['product_id'].duplicated().sum()}")

Number of duplicated products in the dataset is : 0


In [69]:
num_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

display(products_df[num_cols].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]))
#here i found that the minimum value in the product weight is 0, which is impossible so it deserves checking


,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32949.000000,32949.000000,32949.000000,32949.000000
mean,2276.472488,30.815078,16.937661,23.196728
std,4282.038731,16.914458,13.637554,12.079047
min,0.000000,7.000000,2.000000,6.000000
1%,60.000000,16.000000,2.000000,11.000000
25%,300.000000,18.000000,8.000000,15.000000
50%,700.000000,25.000000,13.000000,20.000000
75%,1900.000000,38.000000,21.000000,30.000000
99%,22538.000000,100.000000,69.000000,63.000000
max,40425.000000,105.000000,105.000000,118.000000


In [70]:
display(products_df[products_df['product_weight_g'] == 0 ])

products_with_weight = products_df[products_df['product_weight_g'] > 0]

medians = products_with_weight.groupby('product_category_name')['product_weight_g'].median()
medians['cama_mesa_banho']

columns_to_fill = products_df['product_weight_g'] == 0 
products_df.loc[columns_to_fill , 'product_weight_g'] = medians['cama_mesa_banho']



,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,has_listing_category
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0,True
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0,True
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0,True
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0,True


In [71]:
print((products_df['product_weight_g'] == 0).sum()) 
#now there is no products with impossible weight

0


In [72]:
products_df.rename(columns={'product_name_lenght': 'product_name_length','product_description_lenght' : 'product_description_length'}, inplace=True)
display(products_df.head())
#renamed the columns with the wrong spelling of length 

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,has_listing_category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,True
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,True
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,True
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,True
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,True


In [73]:
#now the translation dataset it translates the product category names from portuguese to english
#i checked for nulls in the first cell

print(f"Number of the duplicated rows : {product_category_name_translation_df.duplicated().sum()}")

unique_columns_translation = set(product_category_name_translation_df['product_category_name'].unique())
print(unique_columns_translation)

unique_columns_category = set(products_df['product_category_name'].unique())
print(unique_columns_translation)

unique_columns_category - unique_columns_translation

# here i checked the duplicates and found 0 duplicated rows
# then i checked the unique values in each set and found 3 values not in the translation dataset , so i will cocat another dataset with these values

Number of the duplicated rows : 0
{'informatica_acessorios', 'brinquedos', 'cds_dvds_musicais', 'artes_e_artesanato', 'cool_stuff', 'relogios_presentes', 'livros_tecnicos', 'fashion_calcados', 'moveis_cozinha_area_de_servico_jantar_e_jardim', 'pcs', 'agro_industria_e_comercio', 'instrumentos_musicais', 'industria_comercio_e_negocios', 'bebes', 'artes', 'musica', 'pet_shop', 'eletronicos', 'cama_mesa_banho', 'climatizacao', 'audio', 'alimentos_bebidas', 'automotivo', 'telefonia', 'casa_conforto', 'perfumaria', 'fashion_underwear_e_moda_praia', 'malas_acessorios', 'moveis_decoracao', 'casa_conforto_2', 'portateis_casa_forno_e_cafe', 'tablets_impressao_imagem', 'consoles_games', 'market_place', 'livros_importados', 'moveis_sala', 'construcao_ferramentas_iluminacao', 'livros_interesse_geral', 'eletrodomesticos_2', 'ferramentas_jardim', 'construcao_ferramentas_jardim', 'beleza_saude', 'moveis_colchao_e_estofado', 'casa_construcao', 'fraldas_higiene', 'seguros_e_servicos', 'cine_foto', 'fash

{'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos', 'unknown'}

In [74]:
extra_translations = pd.DataFrame({'product_category_name': ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos', 'unknown'],
                                   'product_category_name_english': ['pc_gamer', 'food_preparation_appliances' , 'unknown']})

product_category_name_translation_df = pd.concat([product_category_name_translation_df, extra_translations], ignore_index=True)

print(len(product_category_name_translation_df))
product_category_name_translation_df['product_category_name'].duplicated().sum()

#i added the missing translations to the translation dataset and checked for duplicates


74


np.int64(0)

In [ ]:
products_df = products_df.merge(product_category_name_translation_df , on = 'product_category_name', how = 'left')
display(products_df.head())
products_df = products_df[['product_id', 'product_category_name', 'product_category_name_english', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'has_listing_category']]
display(products_df.head())
# Now the products are in the data set with the english translation for the category names

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,has_listing_category,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,True,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,True,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,True,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,True,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,True,housewares


,product_id,product_category_name,product_category_name_english,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,has_listing_category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery,40.0,287.0,1.0,225.0,16.0,10.0,14.0,True
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,True
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure,46.0,250.0,1.0,154.0,18.0,9.0,15.0,True
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby,27.0,261.0,1.0,371.0,26.0,4.0,26.0,True
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares,37.0,402.0,4.0,625.0,20.0,17.0,13.0,True
